In [17]:
from escaperoom_database import  LoadEyeGazeForUsers2,LoadEventForUsers
import pandas as pd
import numpy as np
from itertools import chain
from Util import alphabet
import csv
from sklearn.metrics.pairwise import euclidean_distances

from sklearn.preprocessing import StandardScaler


In [18]:
maxUser = 16
path = 'C:/Users/Administrator/Desktop/VREscapeRoom/EscapeRoomData'
#player_database = LoadFileForUsers(path)
eyegaze_df = LoadEyeGazeForUsers2(path)
event_df = LoadEventForUsers(path)


Openning player0_EyeGazeEvent.csv
Openning player10_EyeGazeEvent.csv
Openning player11_EyeGazeEvent.csv
Openning player12_EyeGazeEvent.csv
Openning player13_EyeGazeEvent.csv
Openning player14_EyeGazeEvent.csv
Openning player1_EyeGazeEvent.csv
Openning player2_EyeGazeEvent.csv
Openning player3_EyeGazeEvent.csv
Openning player4_EyeGazeEvent.csv
Openning player5_EyeGazeEvent.csv
Openning player6_EyeGazeEvent.csv
Openning player7_EyeGazeEvent.csv
Openning player8_EyeGazeEvent.csv
Openning player9_EyeGazeEvent.csv
Openning player0_event.csv
Openning player10_event.csv
Openning player11_event.csv
Openning player12_event.csv
Openning player13_event.csv
Openning player14_event.csv
Openning player1_event.csv
Openning player2_event.csv
Openning player3_event.csv
Openning player4_event.csv
Openning player5_event.csv
Openning player6_event.csv
Openning player7_event.csv
Openning player8_event.csv
Openning player9_event.csv


In [19]:
def RenamePass(df):
    
    #change gaze Object only
    def RenamePass2(df):
        mask = df['GazeObject'].isin(['LoungeChair'])
        df.loc[mask, 'GazeObject'] = 'LoungeChairModel'
        
        mask = df['GazeObject'].isin(['LongTable'])
        df.loc[mask, 'GazeObject'] = 'LongTableModel'
    
    RenamePass2(df)
    objs = ['RedOfficeChair']
    mask = df['ParentName'].isin(objs)
    df.loc[mask, 'ParentName'] = 'ChairGroup'
    
    
    #shelf
    gaze_objects_to_check = ['ShelfSlot1', 'ShelfSlot2', 'ShelfSlot3', 'ShelfSlot4','ShelfSlot5','ShelfSlot6', 'Bookshelf']
    mask = df['ParentName'].isin(gaze_objects_to_check)
    df.loc[mask, 'ParentName'] = 'UpperShelf'
    
    
    objs = ['ShelfSlot7', 'ShelfSlot8', 'ShelfSlot9', 'BookshelfDrawer1','BookshelfDrawer2','BookshelfDrawer3']
    mask = df['GazeObject'].isin(objs)
    df.loc[mask, 'ParentName'] = 'LowerShelf'
    mask2 = df['ParentName'].isin(objs)
    df.loc[mask2, 'ParentName'] = 'LowerShelf'
 

    #cabin
    mask = df['ParentName'].isin(['Cabin'])
    df.loc[mask, 'ParentName'] = 'LowerCabin'
    
    objects_to_check = ['GothicCabinetDoor3', 'GothicCabinetDoor4']
    mask = df['GazeObject'].isin(objects_to_check)
    df.loc[mask, 'ParentName'] = 'LowerCabin'
   
    objects_to_check = ['SmallCabinetDrawer1', 'SmallCabinetDrawer2','SmallCabinetDrawer3']
    mask = df['GazeObject'].isin(objects_to_check)
    mask2 = df['ParentName'].isin(objects_to_check)
    df.loc[mask, 'ParentName'] = 'LoungeChair'
    df.loc[mask2, 'ParentName'] = 'LoungeChair'
    
    
    
    mask = df['ParentName'].isin(['OfficeSet'])
    df.loc[mask, 'ParentName'] = 'OfficeDeskSurface'
    
    #if gaze object is [objects_to_check], then change its parent name
    objects_to_check = ['OfficeTableDrawer1', 'OfficeTableDrawer2','OfficeTableDrawer3','Key5']
    mask = df['GazeObject'].isin(objects_to_check)
    mask2 = df['ParentName'].isin(objects_to_check)
    df.loc[mask, 'ParentName'] = 'OfficeTableDrawer'
    df.loc[mask2, 'ParentName'] = 'OfficeTableDrawer'

    #long table area
    mask = df['ParentName'].isin(['LongTable','ChargingStation'])
    df.loc[mask, 'ParentName'] = 'LongTableSurface'
    
    
    
# add event type to gazedf
def EventPass(gazedf, eventdf):
    gazedf['eventType'] = None  # Initialize the new column

    # Group events by user for efficient lookup
    events_by_user = eventdf.groupby('id')

    for user_id, user_gaze_df in gazedf.groupby('id'):
        if user_id in events_by_user.groups:
            user_events = events_by_user.get_group(user_id)

            for _, event_row in user_events.iterrows():
                event_start = event_row['startFrame']
                event_end = event_row['endFrame']
                event_type = event_row['eventType']

                # Condition 1: gaze interval is within event interval
                # (gaze_start >= event_start AND gaze_end <= event_end)
                condition1 = (user_gaze_df['startKeyframe'] >= event_start) & \
                             (user_gaze_df['endKeyframe'] <= event_end)

                # Condition 2: gaze interval starts within event, but ends after event
                # (gaze_start >= event_start AND gaze_start <= event_end AND gaze_end > event_end)
                condition2 = (user_gaze_df['startKeyframe'] >= event_start) & \
                             (user_gaze_df['startKeyframe'] <= event_end) & \
                             (user_gaze_df['endKeyframe'] > event_end)

                # Apply the event type to rows satisfying either condition
                # We use .loc for safe assignment to a slice of the DataFrame
                gazedf.loc[user_gaze_df.index[condition1 | condition2], 'eventType'] = event_type
    
    return gazedf
    
def Filter_events(df, minDur):
    t = df[df['duration']>= minDur]
    return t
def Merge_events_ObjMerge(df):
    merged_data = []
    pre_chunk_data = None

    for i, row in df.iterrows():
        if pre_chunk_data is None:
            pre_chunk_data = row
        else:
            # If the `chunkName` is the same as the previous one, merge them
            if pre_chunk_data['id'] == row['id'] and pre_chunk_data['AreaName'] == row['AreaName'] and pre_chunk_data['ParentName'] == row['ParentName']:
                pre_chunk_data['endKeyframe'] = row['endKeyframe']
                pre_chunk_data['duration'] += row['duration']
                pre_chunk_data['GazeObject'] = str(pre_chunk_data['GazeObject']) + "," + str(row['GazeObject'])
            else:
                # Append the completed chunk and start a new one
                merged_data.append(pre_chunk_data)
                pre_chunk_data = row
    
     # Don't forget to append the last chunk
    if pre_chunk_data is not None:
        merged_data.append(pre_chunk_data)

    return pd.DataFrame(merged_data)

def ExtractUniqueAreaName(df):
    # Replace "[env]" with "Wall" in the 'AreaName' column
    df['AreaName'] = df['AreaName'].replace('[Env]', 'Wall')
    df['ParentName'] = df['ParentName'].replace('[Env]', 'Wall')
    
    parent_names = df['ParentName'].unique().tolist()
    area_name = df['AreaName'].unique().tolist()
    return parent_names, area_name

# 1. rename some Gazeobject parent area, then add event type to gaze_df
RenamePass(eyegaze_df)
eyegaze_df = EventPass(eyegaze_df, event_df)
# #2. extra unique name list for areaName and Parent Name
parentNameList, areaNameList = ExtractUniqueAreaName(eyegaze_df)
eyegaze_df = Merge_events_ObjMerge(eyegaze_df)
# 
# #3filter out dur < target, and merge again
eyegaze_df =Filter_events(eyegaze_df, 100) 
# # 
eyegaze_df = Merge_events_ObjMerge(eyegaze_df)

eyegaze_df

,id,AreaName,ParentName,GazeObject,startKeyframe,endKeyframe,duration,eventType
34,0,BookShelfArea,UpperShelf,"WoodenShelf,WoodenShelf,book3,book6,book11,boo...",689,2262,1277,0
132,0,BookShelfArea,LowerShelf,"BookshelfDrawer3,book1,book5,book1,book1,book5...",2302,2754,277,0
162,0,OfficeSet,OfficeChairGroup,"Plant3,BookStack6,OfficeChair,BookStack6,Plant...",3094,3477,383,0
179,0,OfficeSet,OfficeDeskSurface,"Key6,OfficeTable,Key6,OpenBook2,OfficeTable,Ke...",3631,3825,194,0
198,0,OfficeSet,OfficeTableDrawer,"OfficeTableDrawer1,BookStack7,OfficeTableDrawe...",3899,4550,583,0
...,...,...,...,...,...,...,...,...
7473,9,OfficeSet,BookStack,"Books3,Books2,Books3,Books2,Books3,Books2,Book...",5009,5323,287,0
7503,9,CabinArea,LowerCabin,"GothicCabinetDoor4,GothicCabinetDoor3,GothicCa...",5445,5636,155,0
7545,9,OfficeSet,OfficeChairGroup,OfficeChair,6241,6366,125,1
7582,9,BookShelfArea,UpperShelf,"WoodenShelf,Books13,WoodenShelf,Bookcase,Paper...",7075,7745,398,1


In [20]:
def AddAreaCode(df_original, isAreaOnly):
    df = df_original.copy() 
    df.loc[:, 'AreaCode'] = None 
    #encode chunk and chunkobj
    for uid, row in df.iterrows():
        AreaName = row['AreaName']
        ParentName = row['ParentName']
        code = None
        if(isAreaOnly):
            code =   str(alphabet[areaNameList.index(AreaName)]) 
        else:
            code =   str(alphabet[areaNameList.index(AreaName)]) +str(parentNameList.index(ParentName))
        df.at[uid, 'AreaCode'] = code
    return df

def ConsturctFeatureSeq(df):
  
    tpd = pd.DataFrame(columns =  ['id', 'sequence']) # initialize a new frame
    
    idList = df['id'].unique()
    for i in range(len(idList)):
        uid = idList[i]
        AreaSeq =  df[(df['id'] == uid)]['AreaCode'].tolist() 
        # find target user, then combine corresponding chunk into list
        row = pd.DataFrame({'id' : uid, 'sequence' : [AreaSeq] })
        tpd = pd.concat([tpd, row], ignore_index= True)

    seen = set()
    Area_alphabets = list(filter(lambda x: not (x in seen or seen.add(x)),
                        chain.from_iterable(tpd['sequence'])))
    print(tpd)
    print(f"encoded select area and  parent obj into alphabet {Area_alphabets};\n   return sequence shape {tpd.shape}")
    
    #sample X sequence(list)
    return tpd, Area_alphabets
def AddExtraFeatureToSeq(df, sequence_df,featNameArr):
    tpd = pd.DataFrame(columns =  ['id', 'duration'])
 
    idList = df['id'].unique()
    for i in range(len(idList)):
        uid = idList[i]
        dur =  np.array(df[df['id'] == uid ]['duration'])
        sumDur = dur.sum()
        stdDur = dur.std()
        avgDur = dur.mean()
        numFixation = len(dur)
        mdict = {}
        #add feature related data to dataframe, then normalize it and add to sequence df
        for feature in featNameArr:
            if(feature == "SumDur"):
                mdict['SumDur'] = sumDur
            if(feature == "StdDur"):
                mdict['StdDur'] = stdDur
            if(feature == "AvgDur"):
                mdict['AvgDur'] = avgDur
            if(feature == "NumFixation"):
                mdict['NumFixation'] = numFixation
        
        mdict['id'] = uid
        mdict['duration'] = [dur]
        row = pd.DataFrame(mdict)
        tpd = pd.concat([tpd, row], ignore_index= True)
    #normalized feature  name
    #normalized value and put that to df
    nfeatureArr = [str('n')+feature for feature in featNameArr] 
    scaler = StandardScaler()
    if(len(featNameArr) != 0):
        X_normalized = scaler.fit_transform(tpd[[feature for feature in featNameArr]])
        array_df= pd.DataFrame(X_normalized, columns = nfeatureArr )
        tpd = pd.concat([tpd, array_df], axis=1)
    
    #merge with sequence df
    tpd = pd.merge(tpd, sequence_df,  on='id', how='left')
    return tpd,nfeatureArr

df = eyegaze_df[eyegaze_df['eventType' ] ==2]
df = AddAreaCode(df, False)
sequence_df, chunk_alphabets = ConsturctFeatureSeq(df)


# featNameArr = [feature for feature, shouldUse in feature_config.items() if shouldUse and feature != "Fixation"]
featNameArr = []
sequence_df,nfeatureArr = AddExtraFeatureToSeq(df, sequence_df,featNameArr)
        

    id                          sequence
0    0  [A0, B2, A0, A1, C8, C6, B4, D7]
1   10                         [D10, A1]
2   11                              [D7]
3   12                      [D7, C8, D7]
4   14             [D10, B4, C8, C6, D7]
5    1                     [C8, D7, D10]
6    2                      [C8, B4, D7]
7    3                      [B4, C8, D7]
8    4              [A0, B4, C6, C8, D7]
9    5         [C5, C6, C5, C6, A1, D10]
10   6                     [C8, C6, D10]
11   7                      [C8, B3, D7]
12   8                      [C5, D7, C9]
13   9                              [D7]
encoded select area and  parent obj into alphabet ['A0', 'B2', 'A1', 'C8', 'C6', 'B4', 'D7', 'D10', 'C5', 'B3', 'C9'];
   return sequence shape (14, 2)


In [21]:
from EyegazeCluster_ER import AppendFeatures,AppendFeatures2,Run_PCA,ConvertSequenceToEmbedding,Construct_HausdorffDistanceMatrix,Construct_KMeanCluster,Construct_AgglomerativeClustering,Construct_Dynmasc,Construct_PAM,EyeGazeClusterMetric,EyeGazeClusterAlgor

num_PCA_comp = 3
# hasFixation = any(shouldUse and feature == "Fixation" 
#                   for feature, shouldUse in feature_config.items()) 
hasFixation = True
if(hasFixation):
    temp_df = sequence_df.copy(deep= True)
    #construct segmentation embedding
    sgtembedding_df = ConvertSequenceToEmbedding(temp_df, chunk_alphabets)
    if(len(featNameArr) != 0):
        sgtembedding_df = AppendFeatures(sgtembedding_df, sequence_df,nfeatureArr)
    print(f"embedding dataset shape {sequence_df.shape}" )

else:
    sgtembedding_df = AppendFeatures2(sequence_df,nfeatureArr)


if(sgtembedding_df.shape[1] >3):
    pca_df,X = Run_PCA(sgtembedding_df, num_PCA_comp)

else:
    pca_df = sgtembedding_df
    X = sgtembedding_df.copy().drop(columns = 'id')
    X = X.to_numpy()

Convert sequence into Embedding; return df num_sampels X embedding_dimension = (14, 122)
embedding dataset shape (14, 3)
Run PCA; df numSample 14 X 3 components; 
sum of PCA 0.58386704674224


In [22]:

# nclusters = int(clusterSetting['NumCluster'])
# clusterAlgo =  clusterSetting['EyeGaze_Algo']
# clusterMetric = clusterSetting['EyeGaze_Metric'] 
nclusters = 3
clusterAlgo= EyeGazeClusterAlgor.Agglomerative.name
clusterMetric = EyeGazeClusterMetric.HausdorffDist.name

distM = None
if(clusterMetric == EyeGazeClusterMetric.Euclidean.name):
    distM = euclidean_distances(X)
else:
    distM = Construct_HausdorffDistanceMatrix(X)

labels = None
if(clusterAlgo ==  EyeGazeClusterAlgor.Agglomerative.name):
    labels,_ = Construct_AgglomerativeClustering(distM, nclusters)
elif (clusterAlgo ==  EyeGazeClusterAlgor.KMean.name):
    labels,_ = Construct_KMeanCluster(X, nclusters)
elif(clusterAlgo ==  EyeGazeClusterAlgor.PAM.name):
    labels,_ = Construct_PAM(X, distM, nclusters)
else:
    print("Error: cannot identify cluster algo; run Kmean by defalt")
    labels,_ = Construct_KMeanCluster(X, nclusters)

In [23]:
def decode_AreaCode(sequence_array, alphabet,areaNameList, parentNameList):
    area_names = []
    parent_names = []
    for encoded_string in sequence_array:
        try:
            # Assuming the first character is the index for alphabet (AreaName)
            # and the remaining characters are the index for parentNameList (ParentName).
            # This assumes that the index for alphabet is always a single digit.
            # If alphabet can have 10 or more items, you'll need a more robust parsing strategy.
            area_char = encoded_string[0]
            parent_index_str = encoded_string[1:]
    
            rea_char_index = alphabet.index(area_char)
            area_name = areaNameList[rea_char_index]
            
            parent_index = int(parent_index_str)
            parent_name = parentNameList[parent_index]
    
            area_names.append(area_name)
            parent_names.append(parent_name)
            
        except (ValueError, IndexError) as e:
            print(f"Error decoding '{encoded_string}': {e}. Check your lists and encoding logic.")
        return area_names, parent_names


    
def outputGroupInfo():
    #adding groupID label back to pca,  then add back to selected chunk seq accordingly
    pca_df['GroupID'] = labels
    for index, row in sequence_df.iterrows():
        uid =  row['id'] 
    
        selectrow =  pca_df[(pca_df['id'] == uid)]
        if not selectrow.empty:
            sequence_df.at[index, 'GroupID'] = selectrow.iloc[0]['GroupID']
        else:
            print("Select row is empty")
    
    # 
    for gi in range(-1, nclusters):
        t = sequence_df[(sequence_df['GroupID'] == gi)]
        selectCol = ['id','sequence']+ nfeatureArr
        
        
    featureArr = [feature for feature in featNameArr] 
    selectCol = ['id', 'GroupID'] + featureArr
    
    df2 = sequence_df[selectCol]
    
    df2 = df2.rename(columns={'id': 'userID'})
    
    for i in range(df2['userID'].count()):
        # if this user does not has a group
        if(df2[df2['userID'] == i].empty) :
            df2.loc[df2.shape[0]] = {'userID': i, 'GroupID' : -1}
    
    df2['dummy'] = 0
    #Save the updated CSV file
    df2.to_csv('../EscapeRoomData/FinalEyeGazeGroup.csv', index=False)
    
    print(df2)
outputGroupInfo()

# df['GazeObject'] = df['GazeObject'].apply(lambda x: str(x) if isinstance(x, (list, np.ndarray)) else x)
df.to_csv('../EscapeRoomData/FinalEyeGazeEvent.csv', index=False, quoting=csv.QUOTE_NONNUMERIC)

   userID  GroupID  dummy
0       0      1.0      0
1      10      0.0      0
2      11      0.0      0
3      12      0.0      0
4      14      0.0      0
5       1      0.0      0
6       2      0.0      0
7       3      0.0      0
8       4      0.0      0
9       5      2.0      0
10      6      0.0      0
11      7      0.0      0
12      8      0.0      0
13      9      0.0      0
14     13     -1.0      0


In [24]:
sequence_df

,id,duration,sequence,GroupID
0,0,"[275, 304, 333, 357, 216, 176, 301, 276]","[A0, B2, A0, A1, C8, C6, B4, D7]",1.0
1,10,"[190, 552]","[D10, A1]",0.0
2,11,[143],[D7],0.0
3,12,"[313, 556, 104]","[D7, C8, D7]",0.0
4,14,"[100, 352, 385, 255, 172]","[D10, B4, C8, C6, D7]",0.0
5,1,"[443, 1460, 113]","[C8, D7, D10]",0.0
6,2,"[684, 103, 664]","[C8, B4, D7]",0.0
7,3,"[137, 187, 137]","[B4, C8, D7]",0.0
8,4,"[909, 108, 186, 135, 201]","[A0, B4, C6, C8, D7]",0.0
9,5,"[142, 136, 2964, 202, 565, 193]","[C5, C6, C5, C6, A1, D10]",2.0


In [25]:
areaNameList

['CabinArea', 'BookShelfArea', 'OfficeSet', 'LongTableArea', 'Wall']

In [26]:
parentNameList

['LowerCabin',
 'LoungeChair',
 'ChairGroup',
 'UpperShelf',
 'LowerShelf',
 'OfficeDeskSurface',
 'OfficeChairGroup',
 'UnderLongTable',
 'OfficeTableDrawer',
 'BookStack',
 'LongTableSurface',
 'Wall']